# Read repertoire evidence against every mock panel

Receptor annotation ranks peptide–MHC candidates. Repertoire inference compares a declared endpoint with matched reference panels. This offline lesson uses **synthetic final-table fixtures**, not biological results or a fitted null. It does not calculate Pgen, draw reference panels or rerun inference.

For your own completed run, replace the fixture paths with the `counts.tsv` and `summary.tsv` exported by `vdjmatch repertoire-exact --pgen-features FEATURES --abundance-metrics`. Keep their producing manifest. The `curve_mean` endpoint measures mean discoverable matched source clonotypes over whole-library sampling effort; it differs from matched-subset Hill diversity, read abundance and asymptotic richness.

The report reuses the shipping `repertoire-report` CLI and retains the recorded empirical P and BH q values. Inspect effect size and all panels, retain every library/timepoint, and distinguish library from donor counts.

In [ ]:
# Report this kernel's versions and prepare a fresh output directory.
import sys, tempfile, subprocess
from pathlib import Path
from importlib.metadata import version
import vdjtools
import polars as pl
print(sys.version.split()[0], {name: version(name) for name in ('vdjmatch','vdjtools','polars')})
output = Path(tempfile.mkdtemp(prefix='vdjmatch-evidence-'))
print(output)

In [ ]:
# Write synthetic final tables; no statistical inference is performed here.
counts = pl.DataFrame({'sample_id': ['DEMO']*201, 'group_id': ['SYNTHETIC_TARGET']*201,
    'replicate': list(range(-1,200)), 'matched_curve_mean': [10.]+[1.]*200})
summary = pl.DataFrame({'sample_id': ['DEMO'], 'sample_label': ['Synthetic example'],
    'group_id': ['SYNTHETIC_TARGET'], 'original_p_curve_mean': [1/201],
    'original_q_curve_mean': [.03]})
counts.write_csv(output/'counts.tsv', separator='\t')
summary.write_csv(output/'summary.tsv', separator='\t')

In [ ]:
# Render all 200 recorded endpoints with the installed CLI.
subprocess.run([sys.executable, '-m', 'vdjmatch.cli', 'repertoire-report',
    str(output/'counts.tsv'), str(output/'summary.tsv'), '--metric', 'curve_mean',
    '--output-prefix', str(output/'evidence')], check=True, timeout=60)
evidence = pl.read_csv(output/'evidence.evidence.tsv', separator='\t')
assert evidence['mock_panels'].to_list() == [200]
assert evidence['recorded_p'][0] == summary['original_p_curve_mean'][0]
assert evidence['recorded_q'][0] == summary['original_q_curve_mean'][0]
evidence

In [ ]:
# Inspect the report; PDF and flat evidence TSV accompany this preview.
from IPython.display import Image, display
display(Image(filename=str(output/'evidence.page-1.png')))

## Interpret the endpoint in its producing context

The illustrative observed/mock ratio is 10; these fabricated values establish only that reporting preserves the input evidence. They make no claim about binding, calibration or biological accuracy.

For biological tables, report the actual matching/V/J predicate, raw marginal Pgen source, observed-control population, count unit, mock count and test family. Full-data discovery uses all observed clone abundances across a fixed whole-library effort grid. The 100,000-read detection experiment is a separate subsampling analysis. Hamming-1 paired exposure is also a distinct workflow: preserve observed pairing and full pMHC/HLA context, and do not transfer the bulk significance law to descriptive paired exposure.

See the [repertoire method guide](https://docs.isalgo.dev/vdjmatch/repertoire-method.html) for the owning algorithms and CLI options.